# 03 · Tabel ASEAN per tahun (D3)

Membuat tabel `asean_tahun` untuk bagian ⑤ *Indonesia vs ASEAN* di dashboard Looker Studio.

| Masukan | Isi | Kegunaan |
|---|---|---|
| `reported-cases-of-measles.csv` | Kasus campak yang dilaporkan (OWID/WHO), sampai 2024 | `kasus_dilaporkan` |
| `who_coverage_mcv1_mcv2.csv` | Cakupan MCV1 dan MCV2 (WHO/UNICEF), kategori **WUENIC** saja | `mcv1_wuenic`, `mcv2_wuenic` |
| `penduduk_asean.csv` | Jumlah penduduk (World Bank `SP.POP.TOTL`) | penyebut `kasus_per_1juta` |
| `pdb_per_kapita_asean.csv` | PDB per kapita US$ (World Bank `NY.GDP.PCAP.CD`) | konteks ekonomi |

**Keputusan yang dipegang** (lihat Rincian Tugas bagian 2.1):
- Kasus WHO/OWID = *kasus dilaporkan*. Tidak dicampur dengan suspek Kemenkes.
- Cakupan hanya kategori WUENIC. ADMIN dan OFFICIAL dibuang.
- Kunci penggabungan: `iso3` + `tahun`. Tahun acuan 2016–2025 (kasus sampai 2024).
- Nilai kosong dibiarkan kosong, tidak diisi 0.

**Keluaran:** `asean_tahun.csv`, siap diunggah ke tab Google Sheets `asean_tahun`.

**Lokasi file.** Keempat file masukan dibaca dari `data/raw/`, hasil `asean_tahun.csv` ditulis ke `data/clean/`.

Jalankan dari atas ke bawah: **Run All**.

## 1. Pengaturan

In [1]:
from pathlib import Path

import pandas as pd

TAHUN_MULAI, TAHUN_AKHIR = 2016, 2025

# 11 negara ASEAN: kode ISO3 -> nama yang dipakai di dashboard
ASEAN = {
    "BRN": "Brunei",
    "KHM": "Kamboja",
    "IDN": "Indonesia",
    "LAO": "Laos",
    "MYS": "Malaysia",
    "MMR": "Myanmar",
    "PHL": "Filipina",
    "SGP": "Singapura",
    "THA": "Thailand",
    "TLS": "Timor-Leste",
    "VNM": "Vietnam",
}

FILE_MASUKAN = [
    "reported-cases-of-measles.csv",
    "who_coverage_mcv1_mcv2.csv",
    "penduduk_asean.csv",
    "pdb_per_kapita_asean.csv",
]

# Struktur repo: masukan dibaca dari data/raw/, keluaran ditulis ke data/clean/.
# Notebook bisa dijalankan dari folder notebooks/ atau dari root repo.
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

kurang = [f for f in FILE_MASUKAN if not (RAW / f).exists()]
assert not kurang, f"File belum lengkap: {kurang}"
print("Membaca dari:", RAW)
print("Menyimpan ke:", CLEAN)

Membaca dari: ..\data\raw
Menyimpan ke: ..\data\clean


## 2. Kasus campak dilaporkan (OWID/WHO)

Kolom aslinya `Entity, Code, Year, Measles - number of reported cases`. Tahun tanpa laporan tidak ada barisnya sama sekali, jadi nanti akan menjadi kosong setelah digabung.

In [2]:
kasus = pd.read_csv(RAW / "reported-cases-of-measles.csv")
kasus.columns = ["negara_owid", "iso3", "tahun", "kasus_dilaporkan"]
kasus = kasus[kasus["iso3"].isin(ASEAN)].drop(columns="negara_owid")

print(kasus.shape)
kasus.query("iso3 == 'IDN' and tahun >= 2019")

(498, 3)


,iso3,tahun,kasus_dilaporkan
4116,IDN,2019,1965
4117,IDN,2020,524
4118,IDN,2021,394
4119,IDN,2022,7704
4120,IDN,2023,18063
4121,IDN,2024,6328


## 3. Cakupan MCV1 dan MCV2 (WHO/UNICEF, WUENIC)

File WHO memuat tiga jenis angka (WUENIC, ADMIN, OFFICIAL). Hanya **WUENIC** yang dipakai. Contoh alasannya: Indonesia 2023 dosis 1, ADMIN 95,41% vs WUENIC 82%.

In [3]:
cov = pd.read_csv(RAW / "who_coverage_mcv1_mcv2.csv")

print("Kategori yang ada:", sorted(cov["COVERAGE_CATEGORY"].unique()))
print("Antigen yang ada :", sorted(cov["ANTIGEN"].unique()))

cov = cov[
    (cov["COVERAGE_CATEGORY"] == "WUENIC")
    & (cov["ANTIGEN"].isin(["MCV1", "MCV2"]))
    & (cov["CODE"].isin(ASEAN))
]

# harus tepat satu angka per negara-tahun-antigen
dup = cov.duplicated(["CODE", "YEAR", "ANTIGEN"]).sum()
assert dup == 0, f"Ada {dup} baris ganda untuk negara-tahun-antigen"

cakupan = (
    cov.pivot(index=["CODE", "YEAR"], columns="ANTIGEN", values="COVERAGE")
    .rename(columns={"MCV1": "mcv1_wuenic", "MCV2": "mcv2_wuenic"})
    .reset_index()
    .rename(columns={"CODE": "iso3", "YEAR": "tahun"})
)
cakupan.columns.name = None

print(cakupan.shape)
cakupan.query("iso3 == 'IDN' and tahun >= 2019")

Kategori yang ada: ['ADMIN', 'OFFICIAL', 'WUENIC']
Antigen yang ada : ['MCV1', 'MCV2']
(484, 4)


,iso3,tahun,mcv1_wuenic,mcv2_wuenic
85,IDN,2019,88.0,71.0
86,IDN,2020,76.0,60.0
87,IDN,2021,72.0,53.0
88,IDN,2022,92.0,71.0
89,IDN,2023,82.0,69.0
90,IDN,2024,85.0,82.0
91,IDN,2025,85.0,78.0


## 4. Penduduk dan PDB per kapita (World Bank)

In [4]:
penduduk = pd.read_csv(RAW / "penduduk_asean.csv")
pdb = pd.read_csv(RAW / "pdb_per_kapita_asean.csv")

# file World Bank juga memuat baris 'Dunia'; ambil hanya 11 negara ASEAN
penduduk = penduduk[penduduk["iso3"].isin(ASEAN)]
pdb = pdb[pdb["iso3"].isin(ASEAN)]

print("Kode di penduduk:", sorted(penduduk["iso3"].unique()))
print("Kode di PDB     :", sorted(pdb["iso3"].unique()))

Kode di penduduk: ['BRN', 'IDN', 'KHM', 'LAO', 'MMR', 'MYS', 'PHL', 'SGP', 'THA', 'TLS', 'VNM']
Kode di PDB     : ['BRN', 'IDN', 'KHM', 'LAO', 'MMR', 'MYS', 'PHL', 'SGP', 'THA', 'TLS', 'VNM']


## 5. Gabungkan dengan kunci `iso3` + `tahun`

Tabel penduduk dipakai sebagai dasar (left join) supaya semua negara dan tahun tetap muncul, walaupun kasus atau cakupannya kosong.

In [5]:
asean = (
    penduduk[["iso3", "tahun", "penduduk"]]
    .merge(kasus, on=["iso3", "tahun"], how="left", validate="one_to_one")
    .merge(cakupan, on=["iso3", "tahun"], how="left", validate="one_to_one")
    .merge(pdb[["iso3", "tahun", "pdb_per_kapita_usd"]],
           on=["iso3", "tahun"], how="left", validate="one_to_one")
)
asean["negara"] = asean["iso3"].map(ASEAN)

# kasus per 1 juta penduduk = kasus / penduduk x 1.000.000
asean["kasus_per_1juta"] = (asean["kasus_dilaporkan"] / asean["penduduk"] * 1_000_000).round(1)

asean = asean[(asean["tahun"] >= TAHUN_MULAI) & (asean["tahun"] <= TAHUN_AKHIR)]

kolom = [
    "iso3", "negara", "tahun", "kasus_dilaporkan", "penduduk",
    "kasus_per_1juta", "mcv1_wuenic", "mcv2_wuenic", "pdb_per_kapita_usd",
]
asean = asean[kolom].sort_values(["iso3", "tahun"]).reset_index(drop=True)

print(asean.shape)
asean.head(12)

(110, 9)


,iso3,negara,tahun,kasus_dilaporkan,penduduk,kasus_per_1juta,mcv1_wuenic,mcv2_wuenic,pdb_per_kapita_usd
0,BRN,Brunei,2016,1.0,427564,2.3,98.0,97.0,26663.30
1,BRN,Brunei,2017,0.0,432772,0.0,97.0,98.0,28024.38
2,BRN,Brunei,2018,1.0,437810,2.3,99.0,98.0,30988.12
3,BRN,Brunei,2019,1.0,442680,2.3,97.0,98.0,30426.57
4,BRN,Brunei,2020,0.0,447404,0.0,99.0,97.0,26834.36
5,BRN,Brunei,2021,0.0,451721,0.0,99.0,99.0,31006.96
6,BRN,Brunei,2022,0.0,455370,0.0,97.0,99.0,36632.93
7,BRN,Brunei,2023,1.0,458949,2.2,97.0,99.0,32890.55
8,BRN,Brunei,2024,0.0,462721,0.0,99.0,99.0,33153.47
9,BRN,Brunei,2025,NaN,466330,NaN,99.0,99.0,32234.64


## 6. Pengecekan

Angka pembanding dari Rincian Tugas D3. Kalau ada yang tidak cocok, cek filter WUENIC dan penyebut penduduknya.

In [6]:
def nilai(iso3, tahun, kolom_):
    return asean.loc[(asean["iso3"] == iso3) & (asean["tahun"] == tahun), kolom_].iloc[0]


cek = [
    ("Indonesia kasus per 1 juta 2023", nilai("IDN", 2023, "kasus_per_1juta"), 64.2),
    ("Indonesia kasus per 1 juta 2024", nilai("IDN", 2024, "kasus_per_1juta"), 22.3),
    ("Indonesia MCV2 2025", nilai("IDN", 2025, "mcv2_wuenic"), 78),
    ("Indonesia MCV1 2025", nilai("IDN", 2025, "mcv1_wuenic"), 85),
    ("Thailand kasus per 1 juta 2024", nilai("THA", 2024, "kasus_per_1juta"), 114.4),
    ("Malaysia kasus per 1 juta 2024", nilai("MYS", 2024, "kasus_per_1juta"), 106.6),
]
hasil = pd.DataFrame(cek, columns=["pengecekan", "hasil", "diharapkan"])
hasil["cocok"] = (hasil["hasil"] - hasil["diharapkan"]).abs() < 0.15
hasil

,pengecekan,hasil,diharapkan,cocok
0,Indonesia kasus per 1 juta 2023,64.2,64.2,True
1,Indonesia kasus per 1 juta 2024,22.3,22.3,True
2,Indonesia MCV2 2025,78.0,78.0,True
3,Indonesia MCV1 2025,85.0,85.0,True
4,Thailand kasus per 1 juta 2024,114.4,114.4,True
5,Malaysia kasus per 1 juta 2024,106.6,106.6,True


In [7]:
assert hasil["cocok"].all(), "Ada angka yang tidak cocok dengan angka pembanding"

assert asean["iso3"].nunique() == 11, "Harus 11 negara"
assert not asean.duplicated(["iso3", "tahun"]).any(), "Ada baris ganda negara-tahun"
assert asean["mcv1_wuenic"].dropna().between(0, 100).all(), "Persen harus 0-100"
assert asean["mcv2_wuenic"].dropna().between(0, 100).all(), "Persen harus 0-100"
print("Semua pengecekan lolos.")

Semua pengecekan lolos.


### Nilai kosong

Kasus kosong untuk tahun 2025 (data WHO baru sampai 2024) dan untuk negara yang tidak melapor (misalnya Singapura 2020–2023). Itu wajar dan tidak diisi 0, karena 0 berarti "tidak ada kasus", bukan "tidak ada data".

In [8]:
kosong = asean.isna().sum().to_frame("jumlah_kosong")
kosong["persen"] = (kosong["jumlah_kosong"] / len(asean) * 100).round(1)
display(kosong)

print("Kasus kosong per negara-tahun:")
asean[asean["kasus_dilaporkan"].isna()].groupby("negara")["tahun"].apply(list)

,jumlah_kosong,persen
iso3,0,0.0
negara,0,0.0
tahun,0,0.0
kasus_dilaporkan,19,17.3
penduduk,0,0.0
kasus_per_1juta,19,17.3
mcv1_wuenic,0,0.0
mcv2_wuenic,1,0.9
pdb_per_kapita_usd,0,0.0


Kasus kosong per negara-tahun:


negara
Brunei                                       [2025]
Filipina                                     [2025]
Indonesia                                    [2025]
Kamboja                                      [2025]
Laos                                         [2025]
Malaysia                                     [2025]
Myanmar                                      [2025]
Singapura      [2016, 2020, 2021, 2022, 2023, 2025]
Thailand                         [2020, 2021, 2025]
Timor-Leste                            [2021, 2025]
Vietnam                                      [2025]
Name: tahun, dtype: object

## 7. Gambaran isi grafik ⑤

Tiga potongan data yang akan dibuat di Looker Studio. Ini hanya untuk memastikan datanya sudah benar. Grafik final dibuat di Looker Studio.

In [9]:
print("MCV2 2025 (batang ⑤a):")
display(asean[asean["tahun"] == 2025][["negara", "mcv1_wuenic", "mcv2_wuenic"]]
        .sort_values("mcv2_wuenic", ascending=False).reset_index(drop=True))

print("Kasus per 1 juta penduduk 2024 (batang ⑤b):")
display(asean[asean["tahun"] == 2024][["negara", "kasus_dilaporkan", "kasus_per_1juta"]]
        .sort_values("kasus_per_1juta", ascending=False).reset_index(drop=True))

print("PDB per kapita 2025 (konteks ⑤c):")
display(asean[asean["tahun"] == 2025][["negara", "pdb_per_kapita_usd"]]
        .sort_values("pdb_per_kapita_usd", ascending=False).reset_index(drop=True))

MCV2 2025 (batang ⑤a):


,negara,mcv1_wuenic,mcv2_wuenic
0,Brunei,99.0,99.0
1,Vietnam,98.0,95.0
2,Singapura,97.0,93.0
3,Thailand,93.0,87.0
4,Malaysia,95.0,84.0
5,Indonesia,85.0,78.0
6,Kamboja,87.0,73.0
7,Myanmar,71.0,68.0
8,Filipina,72.0,67.0
9,Laos,58.0,58.0


Kasus per 1 juta penduduk 2024 (batang ⑤b):


,negara,kasus_dilaporkan,kasus_per_1juta
0,Thailand,8202.0,114.4
1,Malaysia,3791.0,106.6
2,Vietnam,7854.0,77.8
3,Kamboja,666.0,37.8
4,Filipina,3844.0,33.2
5,Indonesia,6328.0,22.3
6,Singapura,12.0,2.0
7,Timor-Leste,2.0,1.4
8,Myanmar,30.0,0.6
9,Laos,1.0,0.1


PDB per kapita 2025 (konteks ⑤c):


,negara,pdb_per_kapita_usd
0,Singapura,98813.98
1,Brunei,32234.64
2,Malaysia,13124.56
3,Thailand,8056.56
4,Vietnam,5065.99
5,Indonesia,5059.63
6,Filipina,4170.72
7,Kamboja,2872.43
8,Laos,2324.76
9,Myanmar,1488.88


## 8. Simpan

Ekspor memakai titik sebagai desimal dan tanpa kolom indeks. Nilai kosong ditulis sebagai sel kosong. Unggah file ini ke tab `asean_tahun` di Google Sheets *Data Dashboard Campak*.

In [10]:
asean.to_csv(CLEAN / "asean_tahun.csv", index=False)
print("Tersimpan:", CLEAN / "asean_tahun.csv")
print(asean.shape)

Tersimpan: ..\data\clean\asean_tahun.csv
(110, 9)
